# Lab 7: Label comments with Gemini

**Allow 30 minutes.** Work in a copy named `lab07_call_gemini_from_python_mine.ipynb`.

In the Week 7 interactive you sent calls to a simulated Gemini. Now you send them to the real Gemini, with your own key: first one question, then a label for each of five comments. The comments are made up.

## 1. Load your key

You need a Gemini API key in the `.env` file in the course repository's top folder. See "API keys" in `SETUP.md`. Keep the key in `.env`, which Git ignores. Do not paste it into this notebook or an AI chat.

Run the cell below. It should print `Key loaded.`

In [ ]:
from pathlib import Path
from datetime import date
import json
import logging
import os
import time

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

for folder in (Path.cwd(), *Path.cwd().parents):
    env_path = folder / ".env"
    if env_path.is_file():
        load_dotenv(env_path, override=True)
        break

key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"

# Hide a warning that the google-genai package prints on every call. It does not mean anything went wrong.
logging.getLogger("google_genai.types").setLevel(logging.ERROR)
print("Key loaded.")

## 2. One call

`client.models.generate_content` sends a prompt to a model. The answer is in `response.text`:

```python
response = client.models.generate_content(model=GEMINI_MODEL, contents="Your question here")
```

Ask Gemini a short question and print `response.text`. Then print `response.usage_metadata`. It shows how many tokens went in and came out.

If you get error 503, the server is busy. Wait a minute and run the cell again.

In [ ]:
# Send one question to Gemini and print response.text.


# Print response.usage_metadata.


## 3. Label one comment

The cell below has five comments and a codebook, like the ones in the interactive, but about a different post. `SYSTEM` is the system instruction: the task and the definition of each label. `LABEL_SCHEMA` lists the fields the answer must have and the labels it may use. `CONFIG` puts both into the settings for a call.

In [ ]:
comments = pd.DataFrame(
    {
        "document_id": ["C1", "C2", "C3", "C4", "C5"],
        "text": [
            "You ate food with their name on it. You owe them an apology and a meal.",
            "Your roommate takes your things all the time. I don't see what you did wrong here.",
            "You shouldn't have eaten it, but shouting at you in front of guests was out of line too.",
            "Oh sure, a name on the box is just a suggestion. Great roommate behavior.",
            "Put a shelf in the fridge for each person and move on.",
        ],
    }
)

SYSTEM = """You label comments on a post. In the post, a person says they ate their roommate's leftovers, which had the roommate's name on the box.

Give the comment one label:
- poster_at_fault: the comment says the poster was in the wrong.
- poster_not_at_fault: the comment says the poster was not in the wrong.
- unclear: the comment does not say who was in the wrong.

Also give a reason of one short sentence."""

ALLOWED_LABELS = ["poster_at_fault", "poster_not_at_fault", "unclear"]

LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "label": {"type": "string", "enum": ALLOWED_LABELS},
        "reason": {"type": "string"},
    },
    "required": ["label", "reason"],
}

CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM,
    response_mime_type="application/json",
    response_schema=LABEL_SCHEMA,
)
comments

**Your turn:**

1. Write a function `build_prompt(text)` that returns the prompt for one comment. Use an f-string, such as `f"Comment: {text}"`. The task and the codebook are already in `SYSTEM`, so the prompt only needs to set out the comment.
2. Make `prompt = build_prompt(text=comments["text"][0])` and print it. This is what Gemini gets, together with `SYSTEM`.
3. Send the prompt with `config=CONFIG` and print `response.text`.
4. `response.text` is a string written as JSON. Turn it into a dictionary with `json.loads(response.text)`, and print the label.

In [ ]:
# Your turn
def build_prompt(text):
    return ...




## 4. Label all five

First, write your own label for each comment, before you see Gemini's labels.

In [ ]:
comments["my_label"] = ["", "", "", "", ""]

Some calls fail. As in the interactive, a 429 (too many calls in one minute) or a 503 (server busy) is temporary, so `ask` below waits and tries again. Any other error stops it right away, because trying again gives the same error. Read the code, then run the cell.

In [ ]:
RETRY_CODES = {429, 503}


def ask(prompt, max_tries=4):
    """Send one prompt with CONFIG. Return the answer as a dictionary. Try again after a temporary error."""
    wait = 10  # seconds to wait before the first retry
    for attempt in range(1, max_tries + 1):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=CONFIG)
            time.sleep(6)  # wait between calls to stay under the free limit of calls per minute
            return json.loads(response.text)
        except errors.APIError as error:
            if error.code not in RETRY_CODES or attempt == max_tries:
                raise  # stop and show the error
            print(f"Error {error.code}. Waiting {wait} seconds, then trying again.")
            time.sleep(wait)
            wait = wait * 2  # wait longer each time: 10, 20, 40 seconds

**Your turn:** finish the loop below. For each comment, build the prompt with `build_prompt(text=comment.text)`, call `ask`, and add the answer to the file right away:

```python
row = pd.DataFrame([{"document_id": comment.document_id, "label": answer["label"], "reason": answer["reason"]}])
row.to_csv(LABELS_PATH, mode="a", header=not LABELS_PATH.exists(), index=False)
```

`mode="a"` adds a line to the end of the file instead of replacing it. Each answer is saved as soon as you get it, so if the loop stops, the answers you already have are safe. The loop skips comments that are already in the file.

Run the cell. Stop it after two or three comments with the stop button, then run it again. It should start where it stopped.

In [ ]:
LABELS_PATH = Path("lab07_labels.csv")
done = set(pd.read_csv(LABELS_PATH)["document_id"]) if LABELS_PATH.exists() else set()

for comment in comments.itertuples():
    if comment.document_id in done:
        continue  # labelled in an earlier run
    # Your turn: build the prompt, call ask, and add one row to LABELS_PATH.


print("Done.")

**Your turn:**

1. Read the file into `results` and put Gemini's labels next to yours (hint: `comments.merge(results, on="document_id")`).
2. Check with `assert` that every label is one of `ALLOWED_LABELS` (hint: `.isin(...).all()`).
3. Where you and Gemini differ, read the comment and Gemini's reason. In a comment, say who you agree with and why.

In [ ]:
# Your turn


## 5. Write down what ran

The same codebook can give other labels with another model or on another day. Add two columns to `results`: the model name (`GEMINI_MODEL`) and today's date (`date.today().isoformat()`). Save the table as `lab07_labels_mine.csv`.

Check that your key does not appear in any cell or output. Then save the notebook.

In [ ]:
# Your turn
